# Vnlp-scale Multi-Model Colab Chat Demo

Select Qwen3.5, Qwen3, Qwen2.5, SmolLM2, Gemma, Llama, Phi, TinyLlama, or a custom Hugging Face model. The notebook uses Vnlp-scale compressed inference for supported models and Transformers for other architectures.

The default is `Qwen/Qwen3.5-0.8B` in text-chat mode. Select **Runtime → Change runtime type → GPU** first.

In [ ]:
import subprocess
import sys
from pathlib import Path

VNLP_REF = "main"
REPO_DIR = Path("/content/Vnlp-scale")
if not REPO_DIR.exists():
    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            VNLP_REF,
            "https://github.com/vnlpscale/Vnlp-scale.git",
            str(REPO_DIR),
        ]
    )
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"{REPO_DIR}[torch,tokenizer]",
        "gradio>=5,<6",
        "accelerate>=1.0",
        "bitsandbytes>=0.45",
        "sentencepiece>=0.2",
        "pillow>=10",
        "torchvision",
        "transformers @ git+https://github.com/huggingface/transformers.git@main",
    ]
)
sys.path.insert(0, str(REPO_DIR / "examples"))
print("Installed the multi-model demo dependencies.")

## Model selection

`CUSTOM_MODEL_ID` overrides the catalog. Use `LOAD_IN_4BIT_OVERRIDE="true"` for larger models on a T4. Gated models require accepting the model license and adding `HF_TOKEN` in Colab Secrets.

In [ ]:
from colab_models import model_profile

MODEL_CHOICE = "Qwen3.5 0.8B"  # @param ["Qwen3.5 0.8B", "Qwen3 0.6B", "Qwen3 1.7B", "Qwen2.5 0.5B Instruct", "SmolLM2 360M Instruct", "SmolLM2 1.7B Instruct", "Gemma 3 1B IT (license required)", "Llama 3.2 1B Instruct (gated)", "Phi-3.5 Mini Instruct 3.8B (4-bit)", "TinyLlama 1.1B Chat (Vnlp-scale compressed)"]
CUSTOM_MODEL_ID = ""  # @param {type:"string"}
CUSTOM_LOADER = "auto"  # @param ["auto", "causal", "multimodal"]
FORCE_BACKEND = "auto"  # @param ["auto", "transformers", "vnlp"]
LOAD_IN_4BIT_OVERRIDE = "profile"  # @param ["profile", "true", "false"]
QUALITY = "low"  # @param ["lossless", "low", "med", "high"]

four_bit = (
    None if LOAD_IN_4BIT_OVERRIDE == "profile" else LOAD_IN_4BIT_OVERRIDE == "true"
)
profile = model_profile(
    MODEL_CHOICE,
    custom_model_id=CUSTOM_MODEL_ID,
    custom_loader=CUSTOM_LOADER,
    force_backend=FORCE_BACKEND,
    load_in_4bit=four_bit,
)
print(profile)

In [ ]:
import os

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN configured:", bool(hf_token))

## Load model

Qwen3.5 uses the latest Transformers multimodal auto-model class, but this chat UI currently sends text only. TinyLlama uses the Vnlp-scale stage-aware compressed runtime.

In [ ]:
from colab_runtime import MultiModelRuntime

runtime = MultiModelRuntime(profile, quality=QUALITY)
print(runtime.load())

warmup, stats = runtime.generate(
    [
        {"role": "system", "content": "You are a concise and helpful assistant."},
        {"role": "user", "content": "Reply with one short greeting."},
    ],
    max_new_tokens=2,
    sample=False,
)
print("Warm-up:", warmup)
print("Stats:", stats)

In [ ]:
from colab_ui import launch_chat

demo = launch_chat(runtime)
demo.launch(share=True, debug=False)

In [ ]:
try:
    demo.close()
except Exception:
    pass
runtime.close()
print("Released the active model runtime.")

## Troubleshooting

- Qwen3.5 requires the latest Transformers build; the installation cell installs it from the official repository.
- For Gemma or Llama, accept the Hugging Face license and configure `HF_TOKEN`.
- For out-of-memory errors, select a smaller model or enable 4-bit loading.
- Custom unsupported models should use the Transformers backend and the correct `CUSTOM_LOADER`.
- To switch models, run Cleanup, change Model selection, then rerun Load and Launch.